In [1]:
import h5py
import numpy as np

#file_path = '/users/2/wan00232/data_folder/Visium_HD/binned_outputs/square_008um/filtered_feature_cell_matrix.h5'
file_path = '/users/2/wan00232/data_folder/Visium_HD/segmented_outputs/filtered_feature_cell_matrix.h5'
f = h5py.File(file_path, 'r')


In [2]:
f['matrix'].keys()

<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>

In [3]:
import pyarrow.parquet as pq
import pandas as pd

# Read the Parquet file into an Arrow Table
table = pq.read_table('/users/2/wan00232/data_folder/Visium_HD/binned_outputs/square_002um/spatial/tissue_positions.parquet')

In [4]:
df_table = table.to_pandas()

In [6]:
import sys
sys.getsizeof(df_table)

1066137664

In [7]:
from scipy.spatial import cKDTree
coords = df_table[['array_row', 'array_col']].values
tree = cKDTree(coords)


In [47]:
tree.query_ball_point([2,0], 1)

def tree_query(rep):
    list_record = []
    for i in range(rep):
        for j in range(rep):
            test_point = tree.query_ball_point([i, j],1)
            df_table.iloc[test_point, :]['in_tissue'].eq(0).any() == np.True_
            list_record.append([i,j])
        
    return #list_record

In [ ]:
import time 

start_time = time.perf_counter()
tree_query(rep = 3349)
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"tree_query executed in: {elapsed_time} seconds")

In [8]:
import numpy as np
# use scipy.spatial.cKDTree when you build `tree` for best speed

def tree_query_edges(rep, tree, df_table, radius=1.0, center_mask=None, return_numpy=True):
    # Precompute a boolean aligned to KD-tree rows: True iff that neighbor is zero-tissue
    is_zero_neighbor = (df_table['in_tissue'].to_numpy() == 0)

    qbp = tree.query_ball_point       # local binding (micro-opt)
    out = []
    out_append = out.append

    if center_mask is not None:
        # iterate only over candidate centers
        ys, xs = np.nonzero(center_mask)
        # iterating over Python ints avoids NumPy scalar overhead in qbp
        for i, j in zip(ys.tolist(), xs.tolist()):
            idxs = qbp([i, j], radius)
            # short-circuit the moment we see a zero neighbor (no temp array)
            for k in idxs:
                if is_zero_neighbor[k]:
                    out_append((i, j))
                    break
    else:
        # full grid (rep x rep)
        for i in range(rep):
            for j in range(rep):
                idxs = qbp([i, j], radius)
                for k in idxs:
                    if is_zero_neighbor[k]:
                        out_append((i, j))
                        break

    return np.asarray(out, dtype=np.int32) if return_numpy else out


In [9]:
import time 

start_time = time.perf_counter()
tree_query_edges(3394, tree, df_table)
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"tree_query executed in: {elapsed_time} seconds")

tree_query executed in: 91.49639195701457 seconds
